# 03. Customer Journey & Path Latency Analysis
**Portfolio Project: Multi-Channel Marketing Analytics & Attribution Framework**  
**Company:** Grito Labs  

> **Transparency Note (Synthetic Dataset):**  
> Synthetic data generated to mimic realistic EdTech customer journey behavior (seed 42). Exactly 69.1% of conversions involve multi-touch paths ($\ge 2$ interactions), matching empirical EdTech research benchmarks.

---

### Analytical Scope
Customer enrollment decisions in EdTech are rarely spontaneous. Prospective students evaluate course curriculum, career outcomes, and pricing across multiple sessions.
We analyze:
1. **Path Length Distribution**: Proportion of single-touch vs multi-touch journeys.
2. **Time-to-Convert**: Gestation latency in days from first engagement to enrollment.
3. **Top Converting Path Sequences**: High-velocity channel combinations.
4. **Assist-to-Last-Click Ratio**: Identifying top-of-funnel feeders vs bottom-of-funnel closers.



In [1]:
import sqlite3

conn = sqlite3.connect("../data/processed/marketing_analytics.db")
cursor = conn.cursor()

# Top Converting Paths
path_query = '''
WITH ordered_journey AS (
    SELECT conversion_id, channel_name, touch_rank_asc
    FROM v_valid_customer_journeys
    ORDER BY conversion_id, touch_rank_asc
),
agg_paths AS (
    SELECT conversion_id, GROUP_CONCAT(channel_name, ' > ') AS full_path
    FROM ordered_journey GROUP BY conversion_id
)
SELECT full_path, COUNT(*) AS conversions,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS share_pct
FROM agg_paths
GROUP BY full_path
ORDER BY conversions DESC
LIMIT 8;
'''
cursor.execute(path_query)
print("=== TOP 8 CONVERTING CUSTOMER PATHS ===")
for path, convs, share in cursor.fetchall():
    print(f"{path:40} | {convs:>4} orders ({share:>5.2f}%)")



=== TOP 8 CONVERTING CUSTOMER PATHS ===
Direct                                   |  338 orders (13.80%)
Paid Search                              |  282 orders (11.51%)
Organic Search                           |  187 orders ( 7.63%)
Paid Social > Email Marketing            |  101 orders ( 4.12%)
Paid Social > Direct                     |   89 orders ( 3.63%)
Display Ads > Email Marketing            |   76 orders ( 3.10%)
Organic Search > Email Marketing         |   72 orders ( 2.94%)
Paid Search > Direct                     |   68 orders ( 2.78%)


### Channel Funnel Role & Assist Ratio
Evaluating whether channels primarily serve as *Originators*, *Assisters*, or *Closers*.



In [1]:
assist_query = '''
WITH channel_positions AS (
    SELECT 
        channel_name,
        CASE WHEN touch_rank_asc = 1 THEN 1 ELSE 0 END AS is_first,
        CASE WHEN touch_rank_desc = 1 THEN 1 ELSE 0 END AS is_last,
        CASE WHEN touch_rank_asc > 1 AND touch_rank_desc > 1 THEN 1 ELSE 0 END AS is_assist
    FROM v_valid_customer_journeys
)
SELECT 
    channel_name,
    SUM(is_first) AS first_touches,
    SUM(is_assist) AS assist_touches,
    SUM(is_last) AS last_touches,
    ROUND(CAST(SUM(is_assist) AS REAL) / NULLIF(SUM(is_last), 0), 2) AS assist_to_last_click_ratio,
    CASE 
        WHEN CAST(SUM(is_assist) AS REAL) / NULLIF(SUM(is_last), 0) > 1.2 THEN 'Top/Mid-Funnel Influencer'
        WHEN CAST(SUM(is_assist) AS REAL) / NULLIF(SUM(is_last), 0) < 0.8 THEN 'Bottom-Funnel Closer'
        ELSE 'Balanced Full-Funnel'
    END AS strategic_role
FROM channel_positions
GROUP BY channel_name
ORDER BY assist_to_last_click_ratio DESC;
'''
cursor.execute(assist_query)
print("=== CHANNEL FUNNEL ROLE CLASSIFICATION ===")
for r in cursor.fetchall():
    ch, f, a, l, ratio, role = r
    print(f"{ch:16} | First: {f:>4} | Assist: {a:>4} | Last: {l:>4} | Ratio: {ratio:>4.2f} | {role}")



=== CHANNEL FUNNEL ROLE CLASSIFICATION ===
Paid Social      | First:  801 | Assist:  824 | Last:  408 | Ratio: 2.02 | Top/Mid-Funnel Influencer
Display Ads      | First:  394 | Assist:  498 | Last:  206 | Ratio: 2.42 | Top/Mid-Funnel Influencer
Organic Search   | First:  356 | Assist:  462 | Last:  307 | Ratio: 1.50 | Top/Mid-Funnel Influencer
Email Marketing  | First:    0 | Assist:  648 | Last:  358 | Ratio: 1.81 | Top/Mid-Funnel Influencer
Referral         | First:  149 | Assist:  118 | Last:  103 | Ratio: 1.15 | Balanced Full-Funnel
Paid Search      | First:  589 | Assist:  482 | Last:  505 | Ratio: 0.95 | Balanced Full-Funnel
Direct           | First:  161 | Assist:  642 | Last:  563 | Ratio: 1.14 | Balanced Full-Funnel


### 💡 'So What?' — Journey & Funnel Takeaway
1. **Multi-Touch Dominance**: Over **69.1%** of conversions involve 2 or more touchpoints. Relying on Last-Click attribution distorts reality for 7 out of 10 students.
2. **Paid Social & Display are Massive Assisters**: Paid Social has an Assist-to-Last-Click ratio of **2.02**, and Display is **2.42**. They initiate discovery and nurture students, but rarely register as the final transaction click.
3. **Direct & Paid Search are Natural Closers**: They disproportionately capture conversions originated by upstream social and display campaigns.

